# Product Recommendation System - Kaggle Dataset Processor

**Objective:** Build a data pipeline that:
1. Reads `20191226-items.csv` (Metadata: Images, Titles, Prices).
2. Reads `20191226-reviews.csv` (Reviews & Ratings).
3. Joins them together using the unique `asin`.
4. Handles the "Cold Start" problem (products with zero reviews).
5. Runs the reviews through a GPU-accelerated NLP Sentiment Model.

### Step 1: Install Dependencies & Setup GPU

In [ ]:
!pip install transformers torch pandas

import pandas as pd
from transformers import pipeline
import torch
from tqdm.auto import tqdm
import json

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

### Step 2: Upload Your Kaggle CSV Files
Make sure you upload `20191226-items.csv` and `20191226-reviews.csv` to the main directory of this Colab notebook using the file explorer on the left.

In [ ]:
import os
if not os.path.exists('20191226-items.csv') or not os.path.exists('20191226-reviews.csv'):
    print("WARNING: Please upload the Kaggle CSV files before proceeding!")
else:
    print("CSV Files found!")

### Step 3: Part A - Load Metadata (Images & Titles)
We load the items and format them for our backend catalog.

In [ ]:
# Load the top 200 phones for our catalog to keep the frontend fast
PRODUCTS_TO_FETCH = 200

print("Loading Items CSV...")
df_items = pd.read_csv('20191226-items.csv').head(PRODUCTS_TO_FETCH)

product_catalog = {}

for index, row in df_items.iterrows():
    asin = str(row['asin'])
    
    title = str(row['title']) if pd.notna(row['title']) else f'Unknown Phone ({asin})'
    image_url = str(row['image']) if pd.notna(row['image']) else "https://via.placeholder.com/400x400?text=No+Image"
    
    # Price handling: sometimes it's '$199.99', sometimes just a number, sometimes missing
    price_str = str(row['price']) if pd.notna(row['price']) else '0'
    try:
        price_val = float(price_str.replace('$', '').replace(',', '').strip())
    except:
        price_val = 0.0
        
    rating = float(row['rating']) if pd.notna(row['rating']) else 0.0
    
    product_catalog[asin] = {
        "id": asin,
        "title": title,
        "image": image_url,
        "price": price_val, 
        "amazon_rating": rating,
        "category": "Cell Phones",
        "reviews": [],
        "has_reviews": False # The Cold Start Flag!
    }

print(f"Successfully built catalog with {len(product_catalog)} products!")

### Step 4: Part B - Load Reviews & Join with Catalog
Now we get the actual reviews based on the ASINs we collected.

In [ ]:
print("Loading Reviews CSV...")
df_reviews = pd.read_csv('20191226-reviews.csv')

reviews_found = 0

for index, row in df_reviews.iterrows():
    asin = str(row['asin'])
    text = str(row['body'])
    rating = float(row['rating']) if pd.notna(row['rating']) else None
    
    if asin in product_catalog and pd.notna(row['body']) and len(text) > 10:
        # WE FOUND A MATCH! Join the review to the metadata product
        product_catalog[asin]['reviews'].append(text[:512]) # Truncate for model limits
        product_catalog[asin]['has_reviews'] = True
        reviews_found += 1
        
print(f"Found {reviews_found} matching reviews for our phones catalog.")

### Step 5: Run the NLP Architecture
Now we calculate sentiment using the Hugging Face Pipeline.

In [ ]:
print("Loading Transformer Model...")
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device)

print("Processing sentiment (This is where the GPU kicks in!)...")
final_dataset = []

for asin, product in tqdm(product_catalog.items()):
    
    # Handle Zero-Review (Cold Start) cases
    if not product['has_reviews'] or len(product['reviews']) == 0:
        product['aiSentimentScore'] = None
        product['features'] = []
        final_dataset.append(product)
        continue
    
    total_ai_score = 0
    all_features = {}
    
    for review in product['reviews']:
        # 1. Run Transformer Sentiment Analysis
        try:
            result = sentiment_analyzer(review[:512], truncation=True)[0]
        except:
            result = {'label': 'POSITIVE', 'score': 0.5}
            
        # 2. Convert to 1-5 Scale
        if result['label'] == 'POSITIVE':
            review_score = 3.0 + (result['score'] * 2.0)
        else:
            review_score = 3.0 - (result['score'] * 2.0)
            
        total_ai_score += review_score
        
        # 3. Simple Keyword Extraction
        keywords = ["camera", "battery", "screen", "design", "performance", "price", "value", "speed"]
        for word in keywords:
            if word in review.lower():
                sentiment_val = 1 if result['label'] == 'POSITIVE' else -1
                all_features[word] = all_features.get(word, 0) + sentiment_val
                
    # 4. Average AI Score
    num_reviews = len(product['reviews'])
    product['aiSentimentScore'] = round(total_ai_score / num_reviews, 2) if num_reviews > 0 else 3.0
    
    # 5. Format Features
    final_features = []
    for name, score_sum in all_features.items():
        final_features.append({
            "name": name,
            "sentiment": 1 if score_sum > 0 else (-1 if score_sum < 0 else 0)
        })
    product['features'] = final_features
    
    final_dataset.append(product)

print("Done!")

### Step 6: Export to JSON
Download this `processed_amazon_data.json` file and place it inside your laptop's `backend/` folder!

In [ ]:
with open('processed_amazon_data.json', 'w') as f:
    json.dump(final_dataset, f, indent=2)

print("Data exported to 'processed_amazon_data.json'. Download it from the files pane on the left, and drop it into your backend folder!")